# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [28]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [29]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # TODO: Implement a tokenization strategy
    B, C, H, W = x.shape
    P = patch_size
    # (B, 1, H//P, P, W//P, P) -> (B, H//P, W//P, P*P) -> (B, num_patches, patch_dim)
    x = x.reshape(B, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)  # (B, H//P, W//P, P, P, C)
    x = x.reshape(B, (H // P) * (W // P), P * P * C)
    return x

In [30]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.linear = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.linear(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.pos_emb = nn.Parameter(torch.randn(1, num_tokens, d_model) * 0.02)
        
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.pos_emb + x
        

In [31]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: implement
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.net(x)


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        # TODO: implement
        self.w_gate = nn.Linear(d_model, d_ff_gated)
        self.w_up = nn.Linear(d_model, d_ff_gated)
        self.w_down = nn.Linear(d_ff_gated, d_model)
        self.drop = nn.Dropout(dropout)

        acts = {
            "glu": nn.Sigmoid(),
            "geglu": nn.GELU(),
            "swiglu": nn.SiLU(),
            "reglu": nn.ReLU(),
        }
        self.gate_act = acts[variant]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        gate = self.gate_act(self.w_gate(x))
        up = self.w_up(x)
        return self.drop(self.w_down(gate * up))

In [32]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        # TODO: implement. For attention use nn.MultiHeadAttention
        self.mlp = mlp
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.drop2 = nn.Dropout(dropout)
         

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        x_norm = self.ln1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.drop1(attn_out)
        return x + self.drop2(self.mlp(self.ln2(x)))

In [34]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> [CLS] + pos embed -> blocks -> CLS output -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        # TODO: implement a strategy for embedding the patches
        self.patch_embedding = PatchEmbed(patch_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embedding = PositionalEmbedding(self.num_tokens + 1, d_model)  # +1 for CLS

        # TODO: implement a strategy to select the right mlp version for your experiment
        def make_mlp():
            if mlp_kind == "ffn":
                return FeedForward(d_model, d_ff, dropout)
            else:
                d_ff_gated = int(2 * d_ff / 3)
                return GLUFeedForward(d_model, d_ff_gated, dropout, variant=mlp_kind)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(), # TODO: Feed your mlp to the encoder blocks
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # TODO: Add a head to project to the amount of output classes you have
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement
        B = x.shape[0]
        patches = patchify(x, self.patch_size)          # (B, num_tokens, patch_dim)
        x = self.patch_embedding(patches)                # (B, num_tokens, d_model)
        cls = self.cls_token.expand(B, -1, -1)           # (B, 1, d_model)
        x = torch.cat([cls, x], dim=1)                   # (B, num_tokens+1, d_model)
        x = self.pos_embedding(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.head(x[:, 0])                      # CLS token output -> (B, 10)
        return logits

In [35]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [36]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb) # TODO: Your criterion

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "mlp_kind": mlp_kind,
        "train_losses": train_losses,
        "test_accs": test_accs,
        "best_test_acc": max(test_accs),
        "final_test_acc": test_accs[-1],
        "n_params": n_params,
    }

In [37]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model config
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ["ffn", "glu", "geglu", "swiglu", "reglu"]
results = []

for kind in runs:
    torch.manual_seed(cfg.seed)
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nRun: {kind} | params: {n_params:,}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

# Print summary
print("\n" + "=" * 60)
print(f"{'Variant':<10} {'Params':>10} {'Best Acc':>10} {'Final Acc':>10}")
print("-" * 60)
for r in results:
    print(f"{r['mlp_kind']:<10} {r['n_params']:>10,} {r['best_test_acc']:>10.4f} {r['final_test_acc']:>10.4f}")


Run: ffn | params: 105,098
[ffn] epoch 1/5 | test acc: 0.8303
[ffn] epoch 2/5 | test acc: 0.8883
[ffn] epoch 3/5 | test acc: 0.9236
[ffn] epoch 4/5 | test acc: 0.9366
[ffn] epoch 5/5 | test acc: 0.9418

Run: glu | params: 105,010
[glu] epoch 1/5 | test acc: 0.8794
[glu] epoch 2/5 | test acc: 0.9274
[glu] epoch 3/5 | test acc: 0.9331
[glu] epoch 4/5 | test acc: 0.9499
[glu] epoch 5/5 | test acc: 0.9541

Run: geglu | params: 105,010
[geglu] epoch 1/5 | test acc: 0.8812
[geglu] epoch 2/5 | test acc: 0.9287
[geglu] epoch 3/5 | test acc: 0.9436
[geglu] epoch 4/5 | test acc: 0.9527
[geglu] epoch 5/5 | test acc: 0.9561

Run: swiglu | params: 105,010
[swiglu] epoch 1/5 | test acc: 0.8731
[swiglu] epoch 2/5 | test acc: 0.9291
[swiglu] epoch 3/5 | test acc: 0.9430
[swiglu] epoch 4/5 | test acc: 0.9475
[swiglu] epoch 5/5 | test acc: 0.9562

Run: reglu | params: 105,010
[reglu] epoch 1/5 | test acc: 0.8766
[reglu] epoch 2/5 | test acc: 0.9213
[reglu] epoch 3/5 | test acc: 0.9444
[reglu] epoch 4/5